## Instalaciones de dependencias

In [1]:
!pip install langchain langchain-mistralai pandas tabulate


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Importación de Librerías

In [3]:
import json
import pandas as pd
from langchain_mistralai import ChatMistralAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
import os

print("Librerías cargadas correctamente")

Librerías cargadas correctamente


## Configurar API Key y modelo

In [4]:
import os
from langchain_mistralai import ChatMistralAI

os.environ["MISTRAL_API_KEY"] = "ttlUH11iq9d1roD58pA3gOlq4DC0zPet"

llm = ChatMistralAI(
    model="mistral-small-latest",
    temperature=0.3
)

# Test rápido
test = llm.invoke("Respondé solo: ¿estás funcionando?")
print(test.content)
print("Mistral conectado correctamente")

Sí, estoy funcionando correctamente. ¿En qué puedo ayudarte?
Mistral conectado correctamente


## Cargar métricas

In [5]:

import json  

with open("metricas.json", "r") as f:
    metricas = json.load(f)  # Cargar el contenido del archivo en un diccionario de Python

print(" Métricas cargadas:")
# Recorrer las métricas y mostrarlas en pantalla, excluyendo los reportes largos de clasificación
for k, v in metricas.items():
    if k not in ["reporte_rf", "reporte_lr"]:
        print(f"  {k}: {v}")


 Métricas cargadas:
  modelo_ganador: RandomForest
  auc_rf: 0.9225
  auc_lr: 0.8771
  cv_rf_media: 0.9153
  cv_lr_media: 0.8646


## Construir el contexto para Mistral

In [6]:
#Construir contexto con las métricas para el prompt
contexto = f"""
RESULTADOS DEL ENTRENAMIENTO — DATASET TELCO CUSTOMER CHURN

Modelo ganador: {metricas['modelo_ganador']}

AUC-ROC:
- Random Forest:       {metricas['auc_rf']}
- Regresión Logística: {metricas['auc_lr']}

Validación Cruzada (AUC media, k=5):
- Random Forest:       {metricas['cv_rf_media']}
- Regresión Logística: {metricas['cv_lr_media']}

Reporte de clasificación — Random Forest:
{metricas['reporte_rf']}

Reporte de clasificación — Regresión Logística:
{metricas['reporte_lr']}
"""

print("Contexto construido")
print(contexto)

Contexto construido

RESULTADOS DEL ENTRENAMIENTO — DATASET TELCO CUSTOMER CHURN

Modelo ganador: RandomForest

AUC-ROC:
- Random Forest:       0.9225
- Regresión Logística: 0.8771

Validación Cruzada (AUC media, k=5):
- Random Forest:       0.9153
- Regresión Logística: 0.8646

Reporte de clasificación — Random Forest:
              precision    recall  f1-score   support

           0       0.86      0.82      0.84      1021
           1       0.83      0.87      0.85      1049

    accuracy                           0.85      2070
   macro avg       0.85      0.85      0.85      2070
weighted avg       0.85      0.85      0.85      2070


Reporte de clasificación — Regresión Logística:
              precision    recall  f1-score   support

           0       0.83      0.73      0.78      1021
           1       0.76      0.86      0.81      1049

    accuracy                           0.79      2070
   macro avg       0.80      0.79      0.79      2070
weighted avg       0.80      0

##  Crear el prompt

In [7]:
#Definir el prompt para el Agente Comunicador
PROMPT_TEMPLATE = """
Sos un analista experto en inteligencia artificial y machine learning.
Tu tarea es generar un reporte ejecutivo claro y profesional en español,
basándote ÚNICAMENTE en los datos del contexto proporcionado.

El reporte debe incluir:
1. Resumen general del problema (predicción de churn en telecomunicaciones)
2. Modelos evaluados y sus métricas principales
3. Modelo ganador y justificación basada en los datos
4. Interpretación del AUC-ROC y la validación cruzada
5. Conclusiones y recomendaciones para la empresa

Contexto:
{context}

Reporte:
"""

prompt = PromptTemplate(
    input_variables=["context"],
    template=PROMPT_TEMPLATE
)

print(" Prompt definido")

 Prompt definido


## Construir la cadena y generar el reporte

In [11]:
# Cadena LangChain y generación del reporte

# Construir la cadena de ejecución: el prompt se pasa al modelo (llm) y luego se procesa la salida como texto
cadena = prompt | llm | StrOutputParser()

print("Generando reporte con Mistral...")
# Invocar la cadena con el contexto previamente definido para obtener el reporte
reporte = cadena.invoke({"context": contexto})

# Mostrar el reporte en formato Markdown dentro del notebook
from IPython.display import display, Markdown
display(Markdown(reporte))


Generando reporte con Mistral...


**Reporte Ejecutivo: Predicción de *Churn* en Telecomunicaciones**
*Análisis de Modelos de Machine Learning para la Retención de Clientes*

---

### **1. Resumen General del Problema**
El *churn* (abandono de clientes) en el sector de telecomunicaciones representa un desafío crítico, con un impacto directo en los ingresos y la sostenibilidad del negocio. En este análisis, se evaluaron dos modelos de *machine learning* para predecir la probabilidad de que un cliente abandone el servicio:
- **Random Forest**
- **Regresión Logística**

El objetivo es identificar el modelo con mayor capacidad predictiva para priorizar acciones de retención (ej.: ofertas personalizadas, soporte proactivo) y reducir costos asociados al *churn*.

---

### **2. Modelos Evaluados y Métricas Principales**

#### **Métricas Clave Comparativas**
| **Modelo**            | **AUC-ROC** | **AUC-ROC (CV, k=5)** | **Precisión (Clase 1)** | **Recall (Clase 1)** | **F1-Score (Clase 1)** | **Exactitud** |
|-----------------------|-------------|-----------------------|-------------------------|----------------------|------------------------|---------------|
| **Random Forest**     | 0.9225      | 0.9153                | 0.83                    | 0.87                 | 0.85                   | 0.85          |
| **Regresión Logística** | 0.8771    | 0.8646                | 0.76                    | 0.86                 | 0.81                   | 0.79          |

**Notas:**
- **AUC-ROC**: Mide la capacidad del modelo para distinguir entre clientes que abandonan (*churn*) y los que no. Valores cercanos a 1 indican mejor desempeño.
- **Recall (Sensibilidad)**: Prioriza la identificación correcta de clientes en riesgo (Clase 1). El Random Forest supera a la Regresión Logística en un 1%.
- **F1-Score**: Balance entre precisión y recall. El Random Forest obtiene un 4% más que la Regresión Logística.

---

### **3. Modelo Ganador y Justificación Basada en Datos**
**Modelo seleccionado:** **Random Forest**
**Razón principal:**
- **Superioridad en métricas clave**:
  - **AUC-ROC (0.9225 vs. 0.8771)**: Indica una mejor capacidad discriminativa, especialmente en umbrales críticos de decisión.
  - **Validación cruzada (0.9153 vs. 0.8646)**: Confirma la robustez del modelo ante variaciones en los datos de entrenamiento.
  - **Recall (0.87)**: Prioriza la detección de clientes en riesgo, esencial para estrategias de retención.
- **Manejo de datos complejos**: El Random Forest es menos sensible a datos desbalanceados y captura relaciones no lineales, comunes en datasets de telecomunicaciones.

---

### **4. Interpretación del AUC-ROC y Validación Cruzada**
#### **AUC-ROC (0.9225 para Random Forest)**
- **Interpretación**: El modelo tiene un 92.25% de probabilidad de clasificar correctamente un par aleatorio de clientes (uno que abandona y otro que no).
- **Implicación**: Alto poder discriminativo, reduciendo falsos positivos/negativos en la identificación de clientes en riesgo.

#### **Validación Cruzada (k=5, AUC medio = 0.9153)**
- **Interpretación**: La consistencia del AUC en los 5 *folds* (0.9153) valida que el modelo no está sobreajustado (*overfitting*) y generaliza bien a datos no vistos.
- **Implicación**: El Random Forest es confiable para implementación en producción.

---

### **5. Conclusiones y Recomendaciones para la Empresa**

#### **Conclusiones**
1. **El Random Forest es la mejor opción** para predecir *churn*, superando a la Regresión Logística en todas las métricas críticas.
2. **Priorización de clientes en riesgo**: Con un *recall* del 87%, el modelo identifica correctamente al 87% de los clientes que abandonarán, permitiendo intervenciones oportunas.
3. **Impacto económico**: Reducir el *churn* en un 10% (ej.: con campañas dirigidas) podría generar ahorros significativos, considerando el *lifetime value* (LTV) de los clientes.

#### **Recomendaciones**
1. **Implementación del modelo**:
   - Desplegar el Random Forest en sistemas de CRM para priorizar clientes con alta probabilidad de *churn*.
   - Integrar alertas automáticas para el equipo de retención (ej.: descuentos, soporte prioritario).

2. **Acciones proactivas**:
   - **Segmentación**: Usar las variables más importantes del modelo (ej.: duración del contrato, uso de datos, soporte técnico) para personalizar estrategias.
   - **Pruebas A/B**: Evaluar el impacto de ofertas específicas en grupos de alto riesgo identificados por el modelo.

3. **Monitoreo continuo**:
   - Reentrenar el modelo cada 3-6 meses con datos actualizados para adaptarse a cambios en el comportamiento del cliente.
   - Monitorear métricas como *precision* y *recall* en producción para ajustar umbrales de decisión.

4. **Enfoque en datos**:
   - Ampliar el dataset con variables adicionales (ej.: satisfacción del cliente, competencia) para mejorar aún más el AUC-ROC.

---
**Nota final**: El Random Forest no solo ofrece un rendimiento superior, sino que también proporciona interpretabilidad (ej.: importancia de variables), clave para alinear las estrategias de negocio con los insights del modelo.

# Guardar y descargar el reporte

In [12]:
# Guardar reporte (VSCode / Jupyter local)
with open("reporte_churn.txt", "w", encoding="utf-8") as f:
    f.write("REPORTE EJECUTIVO — TELCO CUSTOMER CHURN\n")
    f.write("="*60 + "\n\n")
    f.write(reporte)

print(" Reporte guardado como reporte_churn.txt en la carpeta actual")

 Reporte guardado como reporte_churn.txt en la carpeta actual


##  Bot interactivo para consultas

In [16]:
# Celda 9 — Bot interactivo para consultas sobre resultados

# Definir la plantilla del prompt para consultas: el analista responde solo con datos del contexto
PROMPT_CONSULTA = """
Sos un analista de datos experto. Respondé en español usando
ÚNICAMENTE los datos del contexto. Si no está en el contexto,
decí "No tengo esa información en los datos."

Contexto:
{context}

Pregunta: {question}

Respuesta:
"""

# Crear el prompt con variables de entrada (contexto y pregunta)
prompt_consulta = PromptTemplate(
    input_variables=["context", "question"],
    template=PROMPT_CONSULTA
)

# Construir la cadena de ejecución: prompt → modelo → parser
cadena_consulta = prompt_consulta | llm | StrOutputParser()

# Función para realizar consultas interactivas
def consultar(pregunta):
    respuesta = cadena_consulta.invoke({
        "context": contexto,
        "question": pregunta
    })
    print(f"\n {pregunta}")
    print(f" {respuesta}")
    print("─" * 50)

# preguntas al bot
consultar("¿Cuál fue el modelo con mejor rendimiento y por qué?")
consultar("¿Qué indica el AUC-ROC del modelo ganador?")
consultar("¿Qué tan confiable es el modelo según la validación cruzada?")
consultar("¿Qué recomendás para mejorar el modelo?")




 ¿Cuál fue el modelo con mejor rendimiento y por qué?
 El modelo con mejor rendimiento fue **Random Forest**, porque:

1. **AUC-ROC más alto**: 0.9225 (vs. 0.8771 de Regresión Logística).
2. **Mejor validación cruzada**: AUC medio de 0.9153 (vs. 0.8646).
3. **Mejor equilibrio en métricas**:
   - **F1-score**: 0.85 (vs. 0.81 en Regresión Logística).
   - **Recall para la clase 1 (churn)**: 0.87 (vs. 0.86).
   - **Accuracy**: 0.85 (vs. 0.79).

*Nota*: Todas las métricas favorecen a Random Forest en este contexto.
──────────────────────────────────────────────────

 ¿Qué indica el AUC-ROC del modelo ganador?
 El AUC-ROC del modelo ganador (Random Forest) es **0.9225**, lo que indica una excelente capacidad de discriminación entre las clases (clientes que se quedan vs. clientes que se van).
──────────────────────────────────────────────────

 ¿Qué tan confiable es el modelo según la validación cruzada?
 Según la validación cruzada (AUC media, k=5), el modelo **Random Forest** tiene una co